In [ ]:
# Imports and Setup
%matplotlib inline
import subprocess
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from blast_to_qiime import convert_blast_to_qiime
from rdp_to_qiime import rdp_to_qiime_standardized
from merge_utils import merge_and_filter_taxonomy
import shutil


This workflow included Qiime workflow on the cloud for Musquash data for years 2022, 2023 and 2024

In [8]:
# Configurations
root="/mnt/e/projects/Musquash_data/Analysis"
input_dir=f"{root}/data2"
# output_dir=f"{root}/results_diversity_reanalysis/no_prvl_filtering"
output_dir=f"{root}/results2"
os.makedirs(output_dir, exist_ok=True)
metadata_file=f"{root}/metadata/combined_all_year_metada.tsv"
by_station_metadata=f"{root}/metadata/comb_station_grp_all_years_metada.tsv"
# by_station_metadata=f"{root}/metadata/2022_SB_metadata.tsv"
# markers=["mifish"]
# years=["23_m"]
markers=["COI","mifish"]
years=[22, 23, 24, 25]

Convert Blast files into Qiime format

In [ ]:
## BLAST to QIIME Conversion for Multiple Years for MiFish Marker

marker = "COI"
taxonomy = "Blast-MIDORI-90"
# taxonomy = "Blast-nt_euk-90"
SCORING_METHOD = 'evalue_inv'  # For BLAST conversion ('bitscore', 'pident', 'evalue_inv')
NAMES_DMP_PATH = '/mnt/e/projects/databases/tax_dump/ncbi_taxdump/names.dmp'

for year in years:
    
    !qiime tools export --input-path {input_dir}/Musq-{year}-{marker}-rep-seq.qza --output-path {input_dir}/Musq-{year}-{marker}-rep-seq
    !mv {input_dir}/Musq-{year}-{marker}-rep-seq/dna-sequences.fasta {input_dir}/Musq-{year}-{marker}-rep-seq.fasta
    !rm -r {input_dir}/Musq-{year}-{marker}-rep-seq

    QUERY_FASTA_PATH = os.path.join(input_dir, f'Musq-{year}-{marker}-rep-seq.fasta') 
    



    print("--- Starting BLAST Conversion ---")
    BLAST_INPUT = os.path.join(input_dir, f'Musq-{year}-{marker}-{taxonomy}.tsv')
    BLAST_OUTPUT = os.path.join(input_dir, f'Musq-{year}-{marker}-{taxonomy}.qiime')

    if not os.path.exists(BLAST_INPUT) or not os.path.exists(QUERY_FASTA_PATH):
        print("Error: BLAST or FASTA input files not found. Skipping BLAST conversion.")
    else:
        # Call the main function with necessary arguments
        convert_blast_to_qiime(
            blast_path=BLAST_INPUT,
            fasta_path=QUERY_FASTA_PATH,
            output_path=BLAST_OUTPUT,
            names_dmp_path=NAMES_DMP_PATH,
            include_common_names=True,
            use_hierarchical_names=True,
            use_top_scoring=True,  # Use top scoring or consensus logic 
            scoring_method=SCORING_METHOD
        )
        print(f"✅ BLAST output saved to: {BLAST_OUTPUT}")


--- Starting BLAST Conversion ---
Getting lineages via TaxonKit...
Done: /mnt/e/projects/Musquash_data/Analysis/data2/Musq-25_B-COI-Blast-MIDORI-90.qiime
✅ BLAST output saved to: /mnt/e/projects/Musquash_data/Analysis/data2/Musq-25_B-COI-Blast-MIDORI-90.qiime


In [ ]:
# RDP to QIIME Conversion for Multiple Years for COI Marker
# Conversion Parameters
CONFIDENCE_THRESHOLD = 0.9  # For RDP conversion
print("\n--- Starting RDP Conversion ---")
marker = "COI"
taxonomy = "RDP"
# Path to the NCBI taxonomy data (for TaxonKit and common names)
NAMES_DMP_PATH = '/mnt/e/projects/databases/tax_dump/ncbi_taxdump/names.dmp'
for year in years:
    RDP_INPUT = os.path.join(input_dir, f'Musq-{year}-{marker}-{taxonomy}.tsv')
    RDP_OUTPUT = os.path.join(input_dir, f'Musq-{year}-{marker}-{taxonomy}.qiime')

    if not os.path.exists(RDP_INPUT):
        print("Error: RDP input file not found. Skipping RDP conversion.")
    else:
        # Call the main function with necessary arguments
        # It uses the shared NCBITaxonomyLookup for standardization
        rdp_to_qiime_standardized(
            input_file=RDP_INPUT,
            output_file=RDP_OUTPUT,
            names_dmp_path=NAMES_DMP_PATH,
            confidence_threshold=CONFIDENCE_THRESHOLD,
            use_local_db=True,  # Try to use ETE3 DB if available
            include_common_names=True,
            hierarchical_names=True
        )
        print(f"✅ RDP output saved to: {RDP_OUTPUT}")


--- Starting RDP Conversion ---
Parsing RDP file...
Fetching common names for 72 taxa...
Writing output...
✅ RDP output saved to: /mnt/e/projects/Musquash_data/Analysis/data2/Musq-25_B-COI-RDP.qiime


In [ ]:
OBIS_expected_taxa_file = f"{root}/Expected_species/OBIS_expected_species.txt"
Nick_expected_taxa_file = f"{root}/Expected_species/Nick_to_keep_list.txt"
EXPECTED_TAXA_FILE = f"{root}/Expected_species/unique_data.txt"
EXPECTED_TAXA_FILE2 = f"{root}/Expected_species/unique_data2.txt"
SING_EXPECTED_TAXA_FILE = f"{root}/Expected_species/sing_corrected_list.txt"

def merge_expected_taxa(file1, file2, output_file):
    """Read two files into a set and write unique items to output"""
    unique_items = set()
    
    # Read first file
    with open(file1, 'r', encoding='utf-8') as f:
        for line in f:
            unique_items.add(line.strip())  
    
    # Read second file
    with open(file2, 'r', encoding='utf-8') as f:
        for line in f:
            unique_items.add(line.strip())
    
    # Write unique items to output file
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in sorted(unique_items): 
            f.write(item + '\n')


merge_expected_taxa(EXPECTED_TAXA_FILE, SING_EXPECTED_TAXA_FILE, EXPECTED_TAXA_FILE2)

  

In [ ]:
##############################################################
### Function to run Qiime commands conditionally
###############################################################

def run_cmd(cmd, outputs=None, force=False):
    """
    Run a command only if output(s) do not already exist.
    """
    if outputs and not force:
        if all(os.path.exists(o) for o in outputs):
            print(f"✔ Skipping (exists): {outputs}")
            return
    print("▶ Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

##############################################################
### Main Processing Loop
###############################################################
for marker in markers:

    matrices_dir = os.path.join(output_dir, f"{marker}_matrices_input")
    os.makedirs(matrices_dir, exist_ok=True)
    
    for year in years:

        # ===============================
        # Paths
        # ===============================
        sample_filt_table = f"{input_dir}/Musq-{year}-{marker}-neg_cont-filt-table.qza"
        freq_filt_table = f"{input_dir}/Musq-{year}-{marker}-freq-filt-table.qza"
        target_taxa_filt_table = f"{input_dir}/Musq-{year}-{marker}-targtax-filt-table.qza"

        input_rep = f"{input_dir}/Musq-{year}-{marker}-rep-seq.qza"
        freq_filtered_rep = f"{input_dir}/Musq-{year}-{marker}-freq-filt-rep-seq.qza"
        target_taxa_filtered_rep = f"{input_dir}/Musq-{year}-{marker}-targtax-filt-rep-seq.qza"

        # merged_marked_tax_output = f"{input_dir}/Musq-{year}-{marker}-merged-marked-tax.qiime"
        merged_marked_tax_output = f"{input_dir}/Musq-{year}-{marker}-tax.tsv"
        input_taxa = f"{input_dir}/Musq-{year}-{marker}-merged-marked-tax.qza"


        # ================================
        # Marker-specific taxonomy merging
        # ================================
        if marker == "mifish":
            NCBI_file = os.path.join(
                input_dir, f"Musq-{year}-{marker}-Blast-nt_euk-90.qiime"
            )
            MIDORI_80_file = os.path.join(
                input_dir, f"Musq-{year}-mifish-Blast-MIDORI-80.qiime"
            )
            if not os.path.exists(merged_marked_tax_output):
                merge_and_filter_taxonomy(
                    alpha_file=NCBI_file,          # Alpha: NCBI
                    beta_file=MIDORI_80_file,      # Beta: MIDORI
                    lookup_file=EXPECTED_TAXA_FILE2,
                    output_file=merged_marked_tax_output
                )
        
        elif marker == "COI":
            COI_RDP_file = os.path.join(
                input_dir, f"Musq-{year}-{marker}-RDP.qiime"
            )
            COI_BLAST_file = os.path.join(
                input_dir, f"Musq-{year}-{marker}-Blast-nt_euk-90.qiime"
            )
        
            if not os.path.exists(merged_marked_tax_output):
                merge_and_filter_taxonomy(
                    alpha_file=COI_RDP_file,
                    beta_file=COI_BLAST_file,
                    lookup_file=EXPECTED_TAXA_FILE2,
                    output_file=merged_marked_tax_output
                )

        # ===============================
        # Import merged taxonomy
        # ===============================
        run_cmd(
            [
                "qiime", "tools", "import",
                "--type", "FeatureData[Taxonomy]",
                "--input-path", merged_marked_tax_output,
                "--output-path", input_taxa
            ],
            outputs=[input_taxa]
        )

        # ===============================
        # Sample filtering (remove blanks)
        # ===============================
        run_cmd(
            [
                "qiime", "feature-table", "filter-samples",
                "--i-table", f"{input_dir}/Musq-{year}-{marker}-feature-table.qza",
                "--m-metadata-file", metadata_file,
                "--p-where", "[Sample_target] != 'blank'",
                "--o-filtered-table", sample_filt_table
            ],
            outputs=[sample_filt_table], force= True
        )

        # ===============================
        # Frequency filtering
        # ===============================
        run_cmd(
            [
                "qiime", "feature-table", "filter-features",
                "--i-table", sample_filt_table,
                "--p-min-frequency", "5",
                # "--p-min-samples", "2",
                "--o-filtered-table", freq_filt_table
            ],
            outputs=[freq_filt_table], force= True
        )

        # ===============================
        # Taxonomic filtering (Metazoa)
        # ===============================
        run_cmd(
            [
                "qiime", "taxa", "filter-table",
                "--i-table", freq_filt_table,
                "--i-taxonomy", input_taxa,
                "--p-include", "k__Metazoa",
                "--p-exclude", "remove,Aves",
                "--p-mode", "contains",
                "--o-filtered-table", target_taxa_filt_table,
                "--verbose"
            ],
            outputs=[target_taxa_filt_table], force = False
        )

        # ===============================
        # Sequence filtering
        # ===============================
        run_cmd(
            [
                "qiime", "feature-table", "filter-seqs",
                "--i-data", input_rep,
                "--i-table", freq_filt_table,
                "--o-filtered-data", freq_filtered_rep
            ],
            outputs=[freq_filtered_rep], force= True
        )

        run_cmd(
            [
                "qiime", "feature-table", "filter-seqs",
                "--i-data", input_rep,
                "--i-table", target_taxa_filt_table,
                "--o-filtered-data", target_taxa_filtered_rep
            ],
            outputs=[target_taxa_filtered_rep], force = True
        )

        # ===============================
        # Alignment & phylogeny
        # ===============================
        aligned_rep = f"{input_dir}/Musq-{year}-{marker}-filt-alig-rep-seq.qza"
        masked_alig = f"{input_dir}/Musq-{year}-{marker}-filt-alig-msk-rep-seq.qza"
        unrooted_tree = f"{input_dir}/Musq-{year}-{marker}-unrooted-tree.qza"
        rooted_tree = f"{input_dir}/Musq-{year}-{marker}-rooted-tree.qza"

        run_cmd(
            [
                "qiime", "alignment", "mafft",
                "--i-sequences", freq_filtered_rep,
                "--o-alignment", aligned_rep,
                "--verbose"
            ],
            outputs=[aligned_rep], force = False
        )

        run_cmd(
            [
                "qiime", "alignment", "mask",
                "--i-alignment", aligned_rep,
                "--o-masked-alignment", masked_alig,
                "--verbose"
            ],
            outputs=[masked_alig], force = False
        )

        run_cmd(
            [
                "qiime", "phylogeny", "fasttree",
                "--i-alignment", masked_alig,
                "--o-tree", unrooted_tree,
                "--verbose"
            ],
            outputs=[unrooted_tree], force = False
        )

        run_cmd(
            [
                "qiime", "phylogeny", "midpoint-root",
                "--i-tree", unrooted_tree,
                "--o-rooted-tree", rooted_tree,
                "--verbose"
            ],
            outputs=[rooted_tree], force = False
        )


        # ===================================
        # Alpha rarefaction curves(Faith PD)
        # ===================================
        alpha_rarefaction_qzv = (
            f"{input_dir}/Musq-{year}-{marker}-alpha-rarefaction-faithpd.qzv"
        )

        run_cmd(
            [
                "qiime", "diversity", "alpha-rarefaction",
                "--i-table", target_taxa_filt_table,
                "--i-phylogeny", rooted_tree,
                "--p-max-depth", "30000",
                "--p-metrics", "faith_pd",
                "--m-metadata-file", metadata_file,
                "--o-visualization", alpha_rarefaction_qzv
            ],
            outputs=[alpha_rarefaction_qzv]
        )

        # ===============================
        # Grouping & collapsing
        # ===============================
        station_grouped_table = f"{input_dir}/Musq-{year}-{marker}-filt-grpd-station-table.qza"
        station_grouped_table_raw = f"{input_dir}/Musq-{year}-{marker}-filt-grpd-station-raw_table.qza"
        zone_grouped_table = f"{input_dir}/Musq-{year}-{marker}-filt-grpd-zone-table.qza"
        collapsed_table = f"{input_dir}/Musq-{year}-{marker}-filt-grpd-collapsed-table.qza"
        rename_metadata_file = f"{root}/metadata/Musq-{year}-sample-ids-map.tsv"


 
        run_cmd(
            [
                "qiime", "feature-table", "group",
                "--i-table", freq_filt_table,
                "--m-metadata-file", metadata_file,
                "--m-metadata-column", "Station",
                # "--m-metadata-column", "Unique_Water_Sample",
                "--p-mode", "median-ceiling",
                "--p-axis", "sample",
                "--o-grouped-table", station_grouped_table_raw
            ],
            outputs=[station_grouped_table_raw], force= False
        )
                

        run_cmd(
            [
                "qiime", "feature-table", "rename-ids",
                "--i-table", station_grouped_table_raw,
                "--m-metadata-file", rename_metadata_file,
                "--m-metadata-column", "new-id",
                "--p-strict", "True",
                "--o-renamed-table", station_grouped_table
            ],
            outputs=[station_grouped_table], force = False
        )
        station_grouped_table =  station_grouped_table_raw

        run_cmd(
            [
                "qiime", "feature-table", "group",
                "--i-table", target_taxa_filt_table,
                "--m-metadata-file", metadata_file,
                "--m-metadata-column", "Sample_target",
                "--p-mode", "sum",
                "--p-axis", "sample",
                "--o-grouped-table", zone_grouped_table
            ],
            outputs=[zone_grouped_table]
        )


        run_cmd(
            [
                "qiime", "feature-table", "group",
                "--i-table", target_taxa_filt_table,
                "--m-metadata-file", metadata_file,
                "--m-metadata-column", "Zone",
                "--p-mode", "sum",
                "--p-axis", "sample",
                "--o-grouped-table", zone_grouped_table
            ],
            outputs=[zone_grouped_table]
        )

        run_cmd(
            [
                "qiime", "taxa", "collapse",
                "--i-table", station_grouped_table,
                "--i-taxonomy", input_taxa,
                "--p-level", "7",
                "--o-collapsed-table", collapsed_table
            ],
            outputs=[collapsed_table], force= True
        )


        # ===============================
        # Export
        # ===============================
        exported_path = f"{input_dir}/exported-{year}-{marker}-species-table"
        final_species_tsv = f"{input_dir}/Musq-{year}-{marker}-filt-grpd-collapsed-species-table.tsv"

        run_cmd(
            [
                "qiime", "tools", "export",
                "--input-path", f"{input_dir}/Musq-{year}-{marker}-feature-table.qza", #collapsed_table,
                "--output-path", exported_path
            ],
            outputs=[f"{exported_path}/feature-table.biom"], force= True
        )

        run_cmd(
            [
                "biom", "convert",
                "-i", f"{exported_path}/feature-table.biom",
                "-o", final_species_tsv,
                "--to-tsv"
            ],
            outputs=[final_species_tsv], force= True
        )        
        
        
        
        # ===============================
        # Faith PD (alpha diversity)
        # ===============================
        if marker == "mifish":
            RAREF_DEPTH = 10000
        elif marker == "COI":
            RAREF_DEPTH = 20000


        # ===============================
        # Inputs
        # ===============================
        input_table = (
            # collapsed_table
            station_grouped_table
            
        )

        output_core_metrics = (
            f"{output_dir}/Musq-{year}-{marker}-core-metrics-results"
        )
        output_core_metrics_no_phylo = (
            f"{output_core_metrics}/core-metrics-no-phylo"
        )

        output_aitchison = os.path.join(
            output_core_metrics_no_phylo,
            "aitchison_distance_matrix.qza"
        )


        
        # ===============================
        # Core metrics 
        # ===============================
        (phylogenetic)
        run_cmd(
            [
                "qiime", "diversity", "core-metrics-phylogenetic",
                "--i-table", input_table,
                "--i-phylogeny", rooted_tree,
                "--p-sampling-depth", str(RAREF_DEPTH),
                "--m-metadata-file", by_station_metadata,
                "--output-dir", output_core_metrics,
                "--verbose"
            ],
            outputs=[
                f"{output_core_metrics}/faith_pd_vector.qza",
                f"{output_core_metrics}/weighted_unifrac_distance_matrix.qza"
            ], force= False
        )

        # Core metrics (non-phylogenetic)
        run_cmd(
            [
                "qiime", "diversity", "core-metrics",
                "--i-table", input_table,
                "--p-sampling-depth", str(RAREF_DEPTH),
                "--m-metadata-file", by_station_metadata,
                "--output-dir", output_core_metrics_no_phylo,
                "--verbose"
            ],
            outputs=[
                f"{output_core_metrics_no_phylo}/shannon_vector.qza",
                f"{output_core_metrics_no_phylo}/bray_curtis_distance_matrix.qza"
            ], force= False
        )

        # ===============================
        # Alpha diversity group significance
        # ===============================

        # Faith PD
        run_cmd(
            [
                "qiime", "diversity", "alpha-group-significance",
                "--i-alpha-diversity",
                f"{output_core_metrics}/faith_pd_vector.qza",
                "--m-metadata-file", by_station_metadata,
                "--o-visualization",
                f"{output_core_metrics}/faith-pd-group-significance.qzv",
                "--verbose"
            ],
            outputs=[
                f"{output_core_metrics}/faith-pd-group-significance.qzv"
            ]
        )

        # Observed features
        run_cmd(
            [
                "qiime", "diversity", "alpha-group-significance",
                "--i-alpha-diversity",
                f"{output_core_metrics_no_phylo}/observed_features_vector.qza",
                "--m-metadata-file", by_station_metadata,
                "--o-visualization",
                f"{output_core_metrics_no_phylo}/observed-features-group-significance.qzv",
                "--verbose"
            ],
            outputs=[
                f"{output_core_metrics_no_phylo}/observed-features-group-significance.qzv"
            ]
        )

        # Shannon
        run_cmd(
            [
                "qiime", "diversity", "alpha-group-significance",
                "--i-alpha-diversity",
                f"{output_core_metrics_no_phylo}/shannon_vector.qza",
                "--m-metadata-file", by_station_metadata,
                "--o-visualization",
                f"{output_core_metrics_no_phylo}/shannon-group-significance.qzv",
                "--verbose"
            ],
            outputs=[
                f"{output_core_metrics_no_phylo}/shannon-group-significance.qzv"
            ]
        )

        # Aitchison distance matrix 
        run_cmd(
            [
                "qiime", "diversity", "beta",
                "--p-metric", "aitchison",
                "--i-table", input_table,
                "--p-n-jobs", "auto",
                "--o-distance-matrix", output_aitchison,
                "--verbose"
            ],
            outputs=[output_aitchison]
        )



        distance_matrices = {
            "jaccard": os.path.join(
                output_core_metrics_no_phylo,
                "jaccard_distance_matrix.qza"
            ),
            "bray-curtis": os.path.join(
                output_core_metrics_no_phylo,
                "bray_curtis_distance_matrix.qza"
            ),
            "unw-unifrac": os.path.join(
                output_core_metrics,
                "unweighted_unifrac_distance_matrix.qza"
            ),
            "wt-unifrac": os.path.join(
                output_core_metrics,
                "weighted_unifrac_distance_matrix.qza"
            ),
            "aitchison": output_aitchison,
            "shannon": os.path.join(
                output_core_metrics_no_phylo,
                "shannon_vector.qza"
            ),
            "faith-pd": os.path.join(
                output_core_metrics,
                "faith_pd_vector.qza"
            ),
            "observed-features": os.path.join(
                output_core_metrics_no_phylo,
                "observed_features_vector.qza"
            ),
        }

        for label, input_qza in distance_matrices.items():
            if not os.path.exists(input_qza):
                raise FileNotFoundError(f"Missing distance matrix: {input_qza}")

            export_dir = os.path.join(output_core_metrics_no_phylo, f"exported_{label}")

            final_tsv = os.path.join(
                matrices_dir,
                f"Musq-{year}-{marker}-{label}.tsv"
            )

            # qiime tools export
            cmd = [
                "qiime", "tools", "export",
                "--input-path", input_qza,
                "--output-path", export_dir
            ]

            run_cmd(cmd, outputs=[final_tsv])
            # subprocess.run(cmd, check=True)

            if label in ["jaccard", "bray-curtis", "unw-unifrac", "wt-unifrac", "aitchison"]:
                exported_tsv = os.path.join(export_dir, "distance-matrix.tsv")
            else:
                exported_tsv = os.path.join(export_dir, "alpha-diversity.tsv")
            
            if os.path.exists(export_dir):
                if not os.path.exists(exported_tsv):
                    raise FileNotFoundError(f"Expected export not found: {exported_tsv}")


                shutil.move(exported_tsv, final_tsv)

                # Optional cleanup (keeps workspace tidy)
                shutil.rmtree(export_dir)
                print(f"✅ Exported {label} distance matrix to: {final_tsv}")
            elif os.path.exists(final_tsv):
                print(f"skipping : {label} distance matrix exist at : {final_tsv}")



▶ Running: qiime tools export --input-path /mnt/e/projects/Musquash_data/Analysis/data2/Musq-23_m-mifish-feature-table.qza --output-path /mnt/e/projects/Musquash_data/Analysis/data2/exported-23_m-mifish-species-table
Exported /mnt/e/projects/Musquash_data/Analysis/data2/Musq-23_m-mifish-feature-table.qza as BIOMV210DirFmt to directory /mnt/e/projects/Musquash_data/Analysis/data2/exported-23_m-mifish-species-table
▶ Running: biom convert -i /mnt/e/projects/Musquash_data/Analysis/data2/exported-23_m-mifish-species-table/feature-table.biom -o /mnt/e/projects/Musquash_data/Analysis/data2/Musq-23_m-mifish-filt-grpd-collapsed-species-table.tsv --to-tsv


In [ ]:
%load_ext autoreload
%autoreload 2

import species_intersection as si

marker = "COI"
# marker = "mifish"
OUTPUT_DIR = f"{output_dir}/{marker}_species_viz_filtered_modified_singltons"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PREFIX = f"Musq-{marker}"

edit_previous = False
subtask = False  # use to generate the 


if marker == "COI":
    if not subtask:
        EDNA_FILES = {
            f"{input_dir}/Musq-25-{marker}-freq-filt-grpd-collapsed-species-table.tsv": "2025 MS_Ler",
            f"{input_dir}/Musq-24-{marker}-freq-filt-grpd-collapsed-species-table.tsv": "2024 MS_Ler",
            f"{input_dir}/Musq-23-{marker}-freq-filt-grpd-collapsed-species-table.tsv": "2023 NS_LerXT",
            f"{input_dir}/Musq-22-{marker}-freq-filt-grpd-collapsed-species-table.tsv": "2022 NS_LerXT",
        }

        # Historical-record species-name files  →  study label string
        HIST_FILES = {
            f"{root}/Expected_species/Reinhart_2025.txt": "Reinhart et al. (2025)",
            f"{root}/Expected_species/GEO_MBI.txt": "Cooper & Blanchard (2023)",
            f"{root}/Expected_species/singh_all_species_2000.txt": "Singh et al. (2000)",
        }
    
    else:

        OUTPUT_DIR = f"{output_dir}/Musq_22_SB_species_viz"
        PREFIX = f"Musq-{marker}-SB"
        
        EDNA_FILES = {
            f"{output_dir}/Musq-22_SB-{marker}-surface-species-table.tsv": "2022 Surface",
            f"{output_dir}/Musq-22_SB-{marker}-bottom-species-table.tsv": "2022 Bottom",
        }

        HIST_FILES = {
        f"{output_dir}/Musq-22-{marker}-freq-filt-grpd-collapsed-species-table.tsv": "2022 All data",
        }


elif marker == "mifish":
    if not subtask:
        EDNA_FILES = {
            f"{input_dir}/Musq-25-{marker}-freq-filt-grpd-collapsed-species-table.tsv": "2025 MS_MiM",
            f"{input_dir}/Musq-24-{marker}-freq-filt-grpd-collapsed-species-table.tsv": "2024 MS_MiM",
            f"{input_dir}/Musq-23_m-{marker}-freq-filt-grpd-collapsed-species-table.tsv": "2023 MS_MiM",
            f"{input_dir}/Musq-23-{marker}-freq-filt-grpd-collapsed-species-table.tsv": "2023 NS_MiU",
            f"{input_dir}/Musq-22-{marker}-freq-filt-grpd-collapsed-species-table.tsv": "2022 NS_MiU",
        }
        # Historical-record species-name files  →  study label string
        HIST_FILES = {
            f"{root}/Expected_species/Andrew_2021.txt"    : "Cooper et al. (2023)",
            f"{root}/Expected_species/Ipsen_2013.txt"     : "Ipsen et al. (2013)",
            f"{root}/Expected_species/Arens_2007.txt"     : "Arens et al. (2007)",
            f"{root}/Expected_species/singh_fish_2000.txt": "Singh et al. (2000)",
        }
        
    else:
        OUTPUT_DIR = f"{output_dir}/Musq_22_SB_species_viz"
        PREFIX = f"Musq-{marker}-SB"

        EDNA_FILES = {
            f"{output_dir}/Musq-22_SB-{marker}-surface-species-table.tsv": "2022 Surface",
            f"{output_dir}/Musq-22_SB-{marker}-bottom-species-table.tsv": "2022 Bottom",
        }

        HIST_FILES = {
        f"{output_dir}/Musq-22-{marker}-freq-filt-grpd-collapsed-species-table.tsv": "2022 All data",
        }


if not edit_previous:
    STANDARD_ZONES = ["Zone_1", "Zone_2A", "Zone_2B", "Zone_3"]
    master_df, abundance_df = si.merge_and_visualize_species(
        edna_files    = EDNA_FILES,
        hist_files    = HIST_FILES,
        output_dir    = OUTPUT_DIR,
        name_prefix   = PREFIX,
        standard_zones= STANDARD_ZONES,
    )
    print(abundance_df.head(10).to_string(index=False))

else:
    abundance_csv = f"{OUTPUT_DIR}/{PREFIX}_abundance_table.csv"
    si.visualize_species(
        abundance_csv            = abundance_csv,
        edna_files               = EDNA_FILES,
        hist_files               = HIST_FILES,
        output_dir               = OUTPUT_DIR,
        name_prefix              = PREFIX,
        upset_sort_by            = "-degree",
        upset_sort_categories_by = "input",
        abundance_threshold      = 0.001
    )


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loaded 105 species from: /mnt/e/projects/Musquash_data/Analysis/results2/COI_species_viz_filtered_modified/Musq-COI_abundance_table.csv
STEP 5: Generating UpSet plot …


/home/mohamed/miniconda3/envs/qiime2-amplicon-2024.5/lib/python3.9/site-packages/upsetplot/data.py:303: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(False, inplace=True)
/home/mohamed/miniconda3/envs/qiime2-amplicon-2024.5/lib/python3.9/site-packages/upsetplot/plotting.py:795: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on th

Saved: /mnt/e/projects/Musquash_data/Analysis/results2/COI_species_viz_filtered_modified/Musq-COI_upset.png

STEP 6: Generating V & H heatmaps  …
getting hierarchial commonname: {46592, 478208, 190468, 1859082, 187406, 394771, 172567, 3023930, 164412, 251455, 3023941, 119372, 119373, 475726, 51279, 51294, 6759, 134763, 120431, 126592, 37519, 37520, 292503, 314520, 314522, 314529, 37541, 140457, 1903275, 185004, 60593, 195259, 2793148, 2449087, 7671, 6350, 438488, 314080, 3078886, 645360, 1545973, 6390, 2172674, 324873, 1002762, 70924, 371472, 1676567, 36133, 861478, 880429, 880431, 95536, 880433, 95538, 222001, 1582901, 1129270, 2042165, 32570, 40256, 318790, 188241, 869203, 560980, 463189, 529749, 498518, 94552, 59239, 55656, 308606, 1273218, 56195, 587144, 94603, 587148, 1639830, 61334, 6550, 61337, 696728, 526747, 61848, 1886623, 94630, 6063, 97271, 7604, 1037239, 6584, 95170, 1220549, 1514950, 6604, 88015, 2480080, 560612, 97259, 31216, 876016, 31218, 31220, 136183, 2662394}
Saved:

In [67]:
for marker in markers:
  table1 = f"{input_dir}/Musq-22-{marker}-filt-grpd-station-table.qza"
  table2 = f"{input_dir}/Musq-23_m-{marker}-filt-grpd-station-table.qza"
  table3 = f"{input_dir}/Musq-24-{marker}-filt-grpd-station-table.qza"
  table4 = f"{input_dir}/Musq-25-{marker}-filt-grpd-station-table.qza"
  merged_table = f"{input_dir}/Musq-allyear-{marker}-filt-grpd-station-table.qza"

  
  !qiime feature-table merge \
    --i-tables $table1 \
    --i-tables $table2 \
    --i-tables $table3 \
    --i-tables $table4 \
    --o-merged-table $merged_table \
    --verbose

Saved FeatureTable[Frequency] to: /mnt/e/projects/Musquash_data/Analysis/data2/Musq-allyear-mifish-filt-grpd-station-table.qza


In [51]:
# ============================================================
# Export multiple QIIME2 denoising-stats .qza files to one CSV
# Adds columns for:
#   - year
#   - marker
#
# Requirements:
#   pip install qiime2 pandas
#
# Example filenames:
#   2022_12S_denoising-stats.qza
#   2023_COI_denoising-stats.qza
# ============================================================

from pathlib import Path
import pandas as pd
from qiime2 import Artifact
import tempfile
import numpy as np

# ------------------------------------------------------------
# Folder containing the denoising stats .qza files
# ------------------------------------------------------------
# Note: Ensure input_dir and metadata_file variables are defined above this block
stats_dir = Path(f"{input_dir}/denoise_stats/")
print(stats_dir)

# Output CSV file
output_csv = f"{stats_dir}/combined_denoising_stats.csv"
output_summary_csv = f"{stats_dir}/denoising_stats_summary.csv"


# ============================================================
# FIND FILES
# ============================================================
qza_files = list(stats_dir.glob("*.qza"))

print(f"Found {len(qza_files)} qza files")

# ============================================================
# STORAGE
# ============================================================
all_dfs = []

# ============================================================
# PROCESS FILES
# ============================================================
for qza_file in qza_files:

    print("\n----------------------------------")
    print(f"Processing: {qza_file.name}")

    try:

        # ----------------------------------------------------
        # PARSE FILENAME
        # Example:
        # Musq-22-COI-denoise_stats.qza
        # ----------------------------------------------------
        parts = qza_file.stem.split("-")

        year = parts[1]
        marker = parts[2]
        platform = parts[3]

        print(f"Year: {year}")
        print(f"Marker: {marker}")

        # ----------------------------------------------------
        # LOAD ARTIFACT
        # ----------------------------------------------------
        artifact = Artifact.load(str(qza_file))

        print(f"Artifact type: {artifact.type}")

        # ----------------------------------------------------
        # EXPORT ARTIFACT
        # ----------------------------------------------------
        with tempfile.TemporaryDirectory() as tmpdir:

            artifact.export_data(tmpdir)

            # Find TSV file
            tsv_file = list(Path(tmpdir).glob("*.tsv"))[0]

            print(f"Reading: {tsv_file.name}")

            # Read TSV
            df = pd.read_csv(tsv_file, sep="\t")
            # Remove QIIME2 datatype row
            df = df[~df.iloc[:, 0].astype(str).str.startswith("#q2:")]


            # Convert numeric columns
            for col in df.columns[1:]:
                df[col] = pd.to_numeric(df[col], errors="coerce")

        # ----------------------------------------------------
        # ADD METADATA
        # ----------------------------------------------------
        df["Year"] = year
        df["Marker"] = marker
        df["Platform"] = platform

        # ----------------------------------------------------
        # STORE
        # ----------------------------------------------------
        all_dfs.append(df)

        print("Added successfully")

    except Exception as e:

        print(f"ERROR processing {qza_file.name}")
        print(e)

# ============================================================
# MERGE
# ============================================================
print("\n==================================")
print(f"Collected {len(all_dfs)} dataframes")

combined_df = pd.concat(all_dfs, ignore_index=True)


# ============================================================
# Add Metadata (FIXED: Reassigned back to combined_df)
# ============================================================
metadata_df = pd.read_csv(metadata_file, sep="\t")
combined_df = combined_df.merge(metadata_df[['sample-id', 'Sample_target']], on="sample-id", how='left')

# ============================================================
# EXPORT CSV
# ============================================================
combined_df.to_csv(output_csv, index=False)


# ============================================================
# SUMMARY STATISTICS BY YEAR + MARKER
# ============================================================

# Check columns
print(combined_df.columns)

# ============================================================
# NUMERIC COLUMNS ONLY
# ============================================================
numeric_cols = combined_df.select_dtypes(include="number").columns

print("\nNumeric columns:")
print(numeric_cols)

# ============================================================
# SUMMARY STATISTICS
# ============================================================
# ============================================================
# SUMMARY STATISTICS (FIXED FOR CLARITY)
# ============================================================

# Filter out blanks upfront
filtered_df = combined_df[combined_df['Sample_target'] != 'blank']

def format_mean_std(x):
    mean_val = x.mean()
    std_val = x.std()
    if pd.isna(std_val):
        return f"{mean_val:.2f} ± 0.00"
    return f"{mean_val:.2f} ± {std_val:.2f}"

# 1. Calculate Means
means_df = (
    filtered_df
    .groupby(["Marker", 'Platform'])[numeric_cols]
    .agg(format_mean_std)
    .reset_index()
)

# 2. Calculate Medians
medians_df = (
    filtered_df
    .groupby(["Year", "Marker", 'Platform'])[numeric_cols]
    .agg(lambda x: f"{x.median():.2f}")
    .reset_index()
)

# 3. Add a "Statistic" column to distinguish them before stacking
means_df.insert(2, "Statistic", "Mean ± SD")
# medians_df.insert(2, "Statistic", "Median")

# 4. Combine them cleanly
# summary_df = pd.concat([means_df, medians_df], ignore_index=True)
summary_df = means_df
summary_df = summary_df.sort_values(by=["Marker", "Statistic"]).reset_index(drop=True)

# ============================================================
# EXPORT
# ============================================================
summary_df.to_csv(output_summary_csv, index=False)

print("\nSaved summary table:")
print(output_summary_csv)

print(f"\nSaved to: {output_csv}")

/mnt/e/projects/Musquash_data/Analysis/data2/denoise_stats
Found 9 qza files

----------------------------------
Processing: Musq-22-COI-NS-denoise_stats.qza
Year: 22
Marker: COI
Artifact type: SampleData[DADA2Stats]
Reading: stats.tsv
Added successfully

----------------------------------
Processing: Musq-22-mifish-NS-denoise_stats.qza
Year: 22
Marker: mifish
Artifact type: SampleData[DADA2Stats]
Reading: stats.tsv
Added successfully

----------------------------------
Processing: Musq-23-COI-NS-denoise_stats.qza
Year: 23
Marker: COI
Artifact type: SampleData[DADA2Stats]
Reading: stats.tsv
Added successfully

----------------------------------
Processing: Musq-23-mifish-MS-denoise_stats.qza
Year: 23
Marker: mifish
Artifact type: SampleData[DADA2Stats]
Reading: stats.tsv
Added successfully

----------------------------------
Processing: Musq-23-mifish-NS-denoise_stats.qza
Year: 23
Marker: mifish
Artifact type: SampleData[DADA2Stats]
Reading: stats.tsv
Added successfully

--------------